In [1]:
import anndata as ad
import numpy as np
import pandas as pd
from pathlib import Path
import os
import socket
import matplotlib.pyplot as plt
import scanpy as sc
import json
from scipy.sparse import issparse

print(f"Running on node: {socket.gethostname()}")
print(os.getcwd())
print(os.path.expanduser("~"))

Running on node: n-62-20-1
/zhome/bf/7/219671/projects/DL_project
/zhome/bf/7/219671


In [2]:
def get_isoform_proportions(adata, mapping):
    X = adata.X.toarray() if issparse(adata.X) else adata.X
    df_iso = pd.DataFrame(X, columns=adata.var_names, index=adata.obs_names)

    for gene, isoforms in mapping.items():
        valid_isos = [i for i in isoforms if i in df_iso.columns]
        if valid_isos:
            gene_total = df_iso[valid_isos].sum(axis=1)
            df_iso[valid_isos] = df_iso[valid_isos].divide(gene_total + 1e-9, axis=0)

    adata.layers["isoform_proportions"] = df_iso.values.astype(np.float32)
    return adata

In [3]:
def get_gene_mapping(mapping_json_path):
    json_path = Path(mapping_json_path)
    if json_path.exists():
        with open(json_path, 'r') as f:
            return json.load(f)
    else:
        raise FileNotFoundError(f"Mapping file not found at {json_path}")

In [4]:
source_path = Path("/work3/s252608/DL_project/data/raw")

data = ['bulk_processed_genes.h5ad',
        'sc_processed_genes.h5ad']

In [ ]:
for d in data:
    adata = ad.read_h5ad(f'{source_path}/{d}', backed='r')
    print(f'\n{d} loaded')
    sample_slice = adata.X[:100, :100]
    if hasattr(sample_slice, "toarray"):
        sample_slice = sample_slice.toarray()

    print(f"Max value in slice: {np.max(sample_slice)}")
    print(f'log1p: {adata.uns["log1p"]}')

    non_zero = sample_slice[sample_slice > 0]
    print(f"First non-zero values: {non_zero[:5]}")

In [ ]:
exp_data = ad.read_h5ad(f'{source_path}/bulk_processed_genes.h5ad', backed='r')
iso_data = ad.read_h5ad(f'{source_path}/bulk_processed_transcripts.h5ad', backed='r')

In [ ]:
sub_exp_data = exp_data[:1000, :1000].to_memory()
sub_iso_data = iso_data[:1000, :1000].to_memory()

In [ ]:
qc_results = pd.read_csv('/work3/s252608/DL_project/data/qc/bulk_processed_qc_relaxed_labels.csv')
retained_genes = qc_results[qc_results['qc_pass_relaxed']==True]['gene_id'].to_list()
mapping = get_gene_mapping('/work3/s252608/DL_project/data/raw/bulk_processed_gene_to_transcripts.json')

if exp_data.isbacked:
    exp_data = exp_data.to_memory()
    iso_data = iso_data.to_memory()

target_iso_ids = [t for g in retained_genes if g in mapping for t in mapping[g]]
final_iso_ids = [t for t in target_iso_ids if t in iso_data.var_names]

exp_data_sub = exp_data[:, retained_genes].copy()
    
adata_iso_target = iso_data[exp_data_sub.obs_names, final_iso_ids].to_memory()
adata_iso_target = get_isoform_proportions(adata_iso_target.copy(), mapping)

In [ ]:
print(adata_iso_target)

In [ ]:
if "isoform_proportions" in adata_iso_target.layers:
    X_to_plot = adata_iso_target.layers["isoform_proportions"]
else:
    adata_iso_target = get_isoform_proportions(adata_iso_target, mapping)
    X_to_plot = adata_iso_target.layers["isoform_proportions"]

dominant_iso_weights = []
start = 0

for gene, isoforms in mapping.items():
    valid_isos = [i for i in isoforms if i in adata_iso_target.var_names]
    n_iso = len(valid_isos)
    
    if n_iso > 0:
        end = start + n_iso
        max_vals = X_to_plot[:, start:end].max(axis=1)
        dominant_iso_weights.extend(max_vals)
        start = end


plt.figure(figsize=(8, 5))
plt.hist(dominant_iso_weights, bins=50, color='skyblue', edgecolor='black', range=(0, 1))
plt.axvline(x=0.5, color='red', linestyle='--', label='50% Threshold')
plt.title("Proportion of Top Isoform")
plt.xlabel("Proportion (0.0 to 1.0)")
plt.ylabel("Number of Sample-Gene Pairs")
plt.legend()
plt.show()

In [ ]:
sc.pp.log1p(sub_exp_data)
sc.pp.log1p(sub_iso_data)

In [ ]:
nonzero_exp = sub_exp_data.X.data

plt.figure(figsize=(10, 6))
plt.hist(nonzero_exp, bins=100, color='#3498db', edgecolor='black', alpha=0.7)

plt.title('Distribution of Log-Normalized Expression', fontsize=14)
plt.xlabel('log1p', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)

mean_val = np.mean(nonzero_exp)
plt.axvline(mean_val, color='red', linestyle='dashed', linewidth=1.5, label=f'Mean: {mean_val:.2f}')
plt.legend()

plt.show()

In [ ]:
nonzero_exp = sub_iso_data.X.data

plt.figure(figsize=(10, 6))
plt.hist(nonzero_exp, bins=100, color='#3498db', edgecolor='black', alpha=0.7)

plt.title('Distribution of Log-Normalized Isoforms', fontsize=14)
plt.xlabel('log1p', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)

mean_val = np.mean(nonzero_exp)
plt.axvline(mean_val, color='red', linestyle='dashed', linewidth=1.5, label=f'Mean: {mean_val:.2f}')
plt.legend()

plt.show()

In [ ]:
multi_iso_genes = exp_data.uns['multi_isoform_genes']
target_gene = multi_iso_genes[0] 
target_isoforms = exp_data.uns['gene_to_transcripts'][target_gene]

print(f'Comparing target gene {target_gene} with its corresponding isoforms {target_isoforms}')

In [ ]:
gene_vector = exp_data[:, target_gene].to_memory().X.toarray().flatten()

iso_vectors = {}
for iso in target_isoforms:
    if iso in iso_data.var_names:
        iso_vectors[iso] = iso_data[:, iso].to_memory().X.toarray().flatten()

df_comp = pd.DataFrame(iso_vectors)
df_comp['TOTAL_GENE_SUM'] = gene_vector

df_comp_log = np.log1p(df_comp)

In [ ]:
plt.figure(figsize=(12, 6))

df_comp_log.drop(columns=['TOTAL_GENE_SUM']).boxplot()

plt.axhline(df_comp_log['TOTAL_GENE_SUM'].mean(), color='red', linestyle='--', 
            label=f'Mean Total Gene Expression ({target_gene})')

plt.title(f"Isoform Contribution to {target_gene} Expression")
plt.ylabel("log1p Expression")
plt.xlabel("Isoform IDs")
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
def select_hvg(adata, n_top = 5000):
    sc.pp.highly_variable_genes(adata, n_top_genes=n_top)
    return adata[:, adata.var.highly_variable].to_memory()

exp_hvg = select_hvg(exp_data, 5000)

In [ ]:
with open('/work3/s252608/DL_project/data/gene_to_transcripts.json', 'r') as f:
    gene_to_iso = json.load(f)

hvg_names = exp_hvg.var_names.tolist()

all_relevant_iso_ids = []
for gene in hvg_names:
    if gene in gene_to_iso:
        all_relevant_iso_ids.extend(gene_to_iso[gene])

existing_iso_ids = [i for i in all_relevant_iso_ids if i in iso_data.var_names]

iso_hvg = iso_data[:, existing_iso_ids].to_memory()

print(f"Selected {len(hvg_names)} HVGs.")
print(f"Found {len(existing_iso_ids)} associated isoforms as targets.")

In [ ]:
processed_path = Path("/work3/s252608/DL_project/data/processed")
exp_hvg.write_h5ad(processed_path / 'bulk_hvg_normalized.h5ad')